# 01 - Start With Data, Not Metrics

This notebook validates the data foundation for the agentic RAG system before evaluating any model behavior.

Evaluation scope:
- CSV schema and row counts
- Missing value coverage for important fields
- Chroma collection availability and counts
- Basic top-k retrieval smoke tests for both local collections


## Learning Goal

Before writing evals, verify that the data and retrieval substrate are trustworthy enough to evaluate. This lab teaches students to inspect schemas, missing values, collection counts, sample records, and basic retrieval behavior before trusting any metric.

## Where This Fits

Progression: data sanity -> router eval -> retrieval eval -> cascade eval -> answer quality -> full benchmark -> ablation.

This is the "look at the data first" lab. The goal is not to prove the system is good; it is to make sure later eval results are not polluted by broken inputs, missing collections, or malformed records.

## Related AI Evals Concepts

- AI Eval Mistakes: automated evals are weak if you have not looked at the data first.
- Sample Traces: start learning what examples and system records look like before summarizing them.
- When To Write An Eval: avoid writing metrics before you understand the failure surface.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

PROJECT_ROOT


In [ ]:
import importlib

import chromadb
import pandas as pd

import agentic_rag.ingestion as ingestion_module  # noqa: E402
import agentic_rag.retrievers as retrievers_module  # noqa: E402
from agentic_rag.constants import (  # noqa: E402
    DEVICE_COLLECTION,
    QNA_COLLECTION,
    SourceType,
)

importlib.reload(ingestion_module)
importlib.reload(retrievers_module)

from agentic_rag.ingestion import ensure_chroma_collections  # noqa: E402
from agentic_rag.retrievers import ChromaRetriever  # noqa: E402
from agentic_rag.settings import Settings  # noqa: E402
from agentic_rag.telemetry import configure_tracing  # noqa: E402

pd.set_option("display.max_colwidth", 160)


In [ ]:
TRACE_DIR = PROJECT_ROOT / "otel_traces"
TRACE_DIR.mkdir(exist_ok=True)
TRACE_FILE = TRACE_DIR / "01_data_and_chroma_sanity_check.jsonl"

trace_settings = Settings(
    _env_file=None,
    OTEL_TRACING_ENABLED=True,
    OTEL_TRACES_EXPORTER="file",
    OTEL_TRACES_FILE=TRACE_FILE,
    OTEL_SERVICE_NAME="agentic-rag-notebooks",
)
configure_tracing(trace_settings)

TRACE_FILE


## Load Source Data


In [ ]:
qna_path = PROJECT_ROOT / "datasets/medical_qna_dataset.csv"
device_path = PROJECT_ROOT / "datasets/medical_device_manuals_dataset.csv"

qna_df = pd.read_csv(qna_path)
device_df = pd.read_csv(device_path)

dataset_summary = pd.DataFrame(
    [
        {"dataset": "medical_qna_dataset", "rows": len(qna_df), "columns": len(qna_df.columns)},
        {"dataset": "medical_device_manuals_dataset", "rows": len(device_df), "columns": len(device_df.columns)},
    ]
)
dataset_summary


In [ ]:
qna_df.head(3)


In [ ]:
device_df.head(3)


## Check Schema Before Scoring


In [ ]:
expected_qna_columns = {"qtype", "Question", "Answer"}
expected_device_columns = {
    "Device_Name",
    "Model_Number",
    "Manufacturer",
    "Indications_for_Use",
    "Contraindications",
    "Patient_Population",
}

schema_checks = pd.DataFrame(
    [
        {
            "dataset": "qna",
            "missing_expected_columns": sorted(expected_qna_columns - set(qna_df.columns)),
            "status": "pass" if expected_qna_columns <= set(qna_df.columns) else "fail",
        },
        {
            "dataset": "device",
            "missing_expected_columns": sorted(expected_device_columns - set(device_df.columns)),
            "status": "pass" if expected_device_columns <= set(device_df.columns) else "fail",
        },
    ]
)
schema_checks


In [ ]:
def missing_report(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    return pd.DataFrame(
        [
            {
                "column": column,
                "missing_count": int(df[column].isna().sum()),
                "missing_pct": round(float(df[column].isna().mean()), 4),
            }
            for column in columns
        ]
    )

qna_missing = missing_report(qna_df, ["qtype", "Question", "Answer"])
device_missing = missing_report(
    device_df,
    [
        "Device_Name",
        "Model_Number",
        "Manufacturer",
        "Indications_for_Use",
        "Contraindications",
        "Patient_Population",
        "Sterilization_Method",
    ],
)

display(qna_missing)
display(device_missing)


## Verify Retrieval Infrastructure


In [ ]:
settings = Settings(_env_file=None, chroma_path=PROJECT_ROOT / "chroma_db")
client = chromadb.PersistentClient(path=str(settings.chroma_path))

collection_summary = ensure_chroma_collections(client, qna_df, device_df)
collection_summary


## Inspect Stored Documents


In [ ]:
qna_collection = client.get_collection(QNA_COLLECTION)
device_collection = client.get_collection(DEVICE_COLLECTION)

qna_sample = qna_collection.get(limit=2, include=["documents", "metadatas"])
device_sample = device_collection.get(limit=2, include=["documents", "metadatas"])

display(pd.DataFrame({"id": qna_sample["ids"], "document": qna_sample["documents"], "metadata": qna_sample["metadatas"]}))
display(pd.DataFrame({"id": device_sample["ids"], "document": device_sample["documents"], "metadata": device_sample["metadatas"]}))


## Run Retrieval Smoke Tests


In [ ]:

retriever = ChromaRetriever(chroma_path=str(settings.chroma_path), top_k=3)

smoke_queries = [
    {"source": SourceType.RETRIEVE_QNA, "query": "What are the symptoms of dystonia?"},
    {"source": SourceType.RETRIEVE_DEVICE, "query": "What are the contraindications for a dialysis machine?"},
]

async def run_smoke_tests() -> pd.DataFrame:
    rows = []
    for item in smoke_queries:
        docs = await retriever.retrieve(item["source"], item["query"])
        rows.append(
            {
                "source": item["source"].value,
                "query": item["query"],
                "retrieved_count": len(docs),
                "top_doc_id": docs[0].doc_id if docs else None,
                "top_doc_preview": docs[0].text[:240] if docs else None,
                "status": "pass" if docs else "fail",
            }
        )
    return pd.DataFrame(rows)

smoke_results = await run_smoke_tests()
smoke_results


## Summarize Readiness For Evals


In [ ]:
summary = {
    "qna_rows": len(qna_df),
    "device_rows": len(device_df),
    "qna_schema_pass": bool(schema_checks.loc[schema_checks["dataset"] == "qna", "status"].iloc[0] == "pass"),
    "device_schema_pass": bool(schema_checks.loc[schema_checks["dataset"] == "device", "status"].iloc[0] == "pass"),
    "qna_collection_count": int(collection_summary.loc[collection_summary["collection"] == QNA_COLLECTION, "count"].iloc[0]),
    "device_collection_count": int(collection_summary.loc[collection_summary["collection"] == DEVICE_COLLECTION, "count"].iloc[0]),
    "retrieval_smoke_pass": bool((smoke_results["status"] == "pass").all()),
}
summary


## Pay Attention To

- A clean eval starts with boring checks: row counts, column names, nulls, and collection availability.
- Retrieval smoke tests are not final evals; they are early warnings that the substrate may be broken.
- Do not interpret downstream metrics until you know the source data and Chroma collections are aligned.
- Looking at a few records manually is part of the eval process, not a distraction from it.
